In [1]:
import langchain_openai


In [2]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]


In [3]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-4o-mini")


In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [5]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("./LOR.txt", encoding="utf-8")

loaded_data = loader.load()

In [6]:
from langchain_text_splitters import CharacterTextSplitter
text_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
    is_separator_regex=False,
)
texts = text_splitter.create_documents([loaded_data[0].page_content])
text_contents = []
for item in texts:
    text_contents.append(item.page_content)

Created a chunk of size 1855, which is longer than the specified 1000
Created a chunk of size 1787, which is longer than the specified 1000
Created a chunk of size 1861, which is longer than the specified 1000
Created a chunk of size 1292, which is longer than the specified 1000
Created a chunk of size 1371, which is longer than the specified 1000
Created a chunk of size 1298, which is longer than the specified 1000
Created a chunk of size 1076, which is longer than the specified 1000
Created a chunk of size 1740, which is longer than the specified 1000
Created a chunk of size 1041, which is longer than the specified 1000
Created a chunk of size 1512, which is longer than the specified 1000
Created a chunk of size 1544, which is longer than the specified 1000
Created a chunk of size 1505, which is longer than the specified 1000


In [7]:
from langchain_openai import OpenAIEmbeddings
embeddings_model = OpenAIEmbeddings()

In [8]:
from langchain_chroma import Chroma
vector_db = Chroma.from_documents(texts, OpenAIEmbeddings())
retriever = vector_db.as_retriever()

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [10]:
template = """Sana verdiğim  bilgilere dayanarak soruyu kısaca cevapla:
bilgi:
{context}

Soru: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

In [11]:
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

In [12]:
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | chatModel
    | StrOutputParser()
)


In [13]:
response = chain.invoke("who made the one ring")
response

'The One Ring was created by the Dark Lord Sauron.'